#Phase 3 ACR Course Report Summary Table Creation

> This program creates a summary table that is used in Phase 3 ACR course reports. The summary table will display in the final cell of this notebook. No tables in the Databricks catalog are altered by this script. This script is purely for generating calculations for the Phase 3 ACR reports.

> The data comes from two surveys in Qualtrics that are managed by the PECQI unit.
>   - End of ACR Survey - All ACRs except Medicine
>   - End of ACR Survey - Medicine

> Specify the parameters in the widgets above. Simply use "Internal Medicine" to specify the IM ACR. Otherwise, see the sandbox.pecqi.p3_course_code_map_prod table for ACR names.

> When specifying the _Start Academic Year_ and _Start Block_, you should use the more recent academic year if the block overlaps another cohort. For example, AY 23-24 Block 16 and AY 24-25 Block 4 occur at the same time, therefore, you would specify the most recent academic year and block combination, which in this case is AY 24-25 Block 4. Likewise, use the more recent ay-block combination when specifying _End Academic Year_ and _End Block_. By specifying the Start Academic Year and Start Block in this manner, the program appropriately includes and excludes overlapping blocks. Review the output from the ay_block list below to see the combinations of ay blocks that were included in the analysis after running the script.

##Access Parameter Values

In [0]:
# Access the parameters from the widgets
acr = dbutils.widgets.get("acr")
end_ay = dbutils.widgets.get("end_ay")
end_block = dbutils.widgets.get("end_block")
start_ay = dbutils.widgets.get("start_ay")
start_block = dbutils.widgets.get("start_block")

##Generate Dataframe Containing AY-Block Pairs to Evaluate


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Define the function to generate the academic year-block combinations to be analyzed
def generate_ay_blocks(start_ay, end_ay, start_block, end_block):
    # Convert block numbers to integers
    start_block_num = int(start_block.split(" ")[1])
    end_block_num = int(end_block.split(" ")[1])
    print(f"start_block_num: {start_block_num}, end_block_num: {end_block_num}") 

    # Extract academic years as integers (assuming format like "23-24")
    start_year1, start_year2 = map(int, start_ay.split("-"))
    end_year1, end_year2 = map(int, end_ay.split("-"))
    print(f"start_year1: {start_year1}, start_year2: {start_year2}, end_year1: {end_year1}, end_year2: {end_year2}")

    ay_blocks = []

    # Include overlapping blocks from the previous academic year
    if start_block_num <= 4:
        previous_start_year1 = start_year1 - 1
        previous_start_year2 = start_year2 - 1
        print(f"Previous academic year: {previous_start_year1}-{previous_start_year2}")

        for block in range(13 + (start_block_num-1), 17):
            block_entry = (f"Block {block}", f"{previous_start_year1:02d}-{previous_start_year2:02d}")
            ay_blocks.append(block_entry)
            print(f"Added from previous year: {block_entry}")

    # Include blocks within the range of start and end academic years
    current_year1 = start_year1
    current_year2 = start_year2

    for year1 in range(start_year1, end_year1 + 1):
        for block in range(1, 17):
            if (year1 == start_year1 and block < start_block_num) or (year1 == end_year1 and block > end_block_num):
                continue

            block_entry = (f"Block {block}", f"{current_year1:02d}-{current_year2:02d}")
            ay_blocks.append(block_entry)
            print(f"Added: {block_entry}")

        current_year1 += 1
        current_year2 += 1

    # Remove blocks that should not be included due to overlapping blocks of the previous academic year
    final_year_records_to_exclude = set()
    if end_block_num <= 4:
        previous_end_year1 = end_year1 - 1
        previous_end_year2 = end_year2 - 1
        for block in range(17 - (4 - end_block_num), 17):
            block_entry = (f"Block {block}", f"{previous_end_year1:02d}-{previous_end_year2:02d}")
            final_year_records_to_exclude.add(block_entry)
            print(f"Exclusion added: {block_entry}")

    ay_blocks = [x for x in ay_blocks if x not in final_year_records_to_exclude]

    return ay_blocks

# Generate the data
data = generate_ay_blocks(start_ay, end_ay, start_block, end_block)

# Define schema for the DataFrame
schema = StructType([
    StructField("block", StringType(), True),
    StructField("academic_year", StringType(), True)
])

# Create the DataFrame
ay_blocks_df = spark.createDataFrame(data, schema)

# If needed, collect the data to the driver for further use (e.g., in pandas)
ay_blocks = ay_blocks_df.collect()

In [0]:
# view ay-blocks pairs to be included in the summary table
display(ay_blocks)

##Import Data

In [0]:
# Read the medicine acr table into a DataFrame
df_acr_med = spark.table("sandbox.pecqi.p3_acr_med_tbl_prod")

# Read the non-medicine acr table into a DataFrame
df_acr_nonmed = spark.table("sandbox.pecqi.p3_acr_nonmed_tbl_prod")

##Recode and Transform Data

####Medicine ACR Data

In [0]:
# Create a list of column names to be recoded
med_col_labels = [
    "overall_qual_day", "overall_qual_night", "workload_day", 
    "workload_night", "teach_faculty_day", "teach_res_day", 
    "teach_faculty_night", "teach_res_night"
]

# Create a list of new column names to hold the numeric values
med_recode_labels =  [
    "overall_qual_day_num", "overall_qual_night_num", "workload_day_num", 
    "workload_night_num", "teach_faculty_day_num", "teach_res_day_num", 
    "teach_faculty_night_num", "teach_res_night_num"
]

In [0]:
from pyspark.sql.functions import col, when

# Create new numeric columns for each of the specified columns based on satisfaction level
for col_label, recode_label in zip(med_col_labels, med_recode_labels):
  df_acr_med = df_acr_med.withColumn(
    recode_label,
    when(col(col_label) == "Very Satisfied", 5)
    .when(col(col_label) == "Satisfied", 4)
    .when(col(col_label) == "Neutral", 3)
    .when(col(col_label) == "Dissatisfied", 2)
    .when(col(col_label) == "Very Dissatisfied", 1)
    .otherwise(None)  # Use None for unmatched entries to represent missing values
)

In [0]:
# Collapse day and night columns into a single column while accounting for null values
from pyspark.sql.functions import col, when, isnull

# Calculate overall_qual_num
df_acr_med = df_acr_med.withColumn(
    "overall_qual_num",
    when(isnull(col("overall_qual_day_num")) & isnull(col("overall_qual_night_num")), None) #when both columns are null, set to null
    .when(isnull(col("overall_qual_day_num")), col("overall_qual_night_num")) #when only one column is null, set to the other column
    .when(isnull(col("overall_qual_night_num")), col("overall_qual_day_num")) #when only one column is null, set to the other column
    .otherwise((col("overall_qual_day_num") + col("overall_qual_night_num")) / 2) #when neither column is null, set to the average of the two columns
)

# Calculate workload_num
df_acr_med = df_acr_med.withColumn(
    "workload_num",
    when(isnull(col("workload_day_num")) & isnull(col("workload_night_num")), None) #when both columns are null, set to null
    .when(isnull(col("workload_day_num")), col("workload_night_num")) #when only one column is null, set to the other column
    .when(isnull(col("workload_night_num")), col("workload_day_num")) #when only one column is null, set to the other column
    .otherwise((col("workload_day_num") + col("workload_night_num")) / 2) #when neither column is null, set to the average of the two columns
)

In [0]:
# Calculate teach_num
# Collapse day-night and faculty-resident teach columns into a single column while accounting for null values
from pyspark.sql.functions import col, when, isnull, expr

# Calculate the sum of non-NULL columns
df_acr_med = df_acr_med.withColumn(
    "teach_sum",
    when(isnull(col("teach_faculty_day_num")), 0).otherwise(col("teach_faculty_day_num")) + #when null set to 0, otherwise set to the column value
    when(isnull(col("teach_faculty_night_num")), 0).otherwise(col("teach_faculty_night_num")) + #when null set to 0, otherwise set to the column value
    when(isnull(col("teach_res_day_num")), 0).otherwise(col("teach_res_day_num")) + #when null set to 0, otherwise set to the column value
    when(isnull(col("teach_res_night_num")), 0).otherwise(col("teach_res_night_num")) #when null set to 0, otherwise set to the column value
)

# Count the number of non-NULL columns
df_acr_med = df_acr_med.withColumn(
    "teach_count",
    (when(isnull(col("teach_faculty_day_num")), 0).otherwise(1)) + #when null set to 0, otherwise add 1
    (when(isnull(col("teach_faculty_night_num")), 0).otherwise(1)) + #when null set to 0, otherwise add 1
    (when(isnull(col("teach_res_day_num")), 0).otherwise(1)) + #when null set to 0, otherwise add 1
    (when(isnull(col("teach_res_night_num")), 0).otherwise(1)) #when null set to 0, otherwise add 1
)

# Calculate teach_num by dividing the sum by the count (only if there are non-NULL values)
df_acr_med = df_acr_med.withColumn(
    "teach_num",
    when(col("teach_count") == 0, None) #if all columns were null (represented by 0), set to null
    .otherwise(col("teach_sum") / col("teach_count")) #otherwise, divide the sum by the count
)

# Drop intermediate columns
df_acr_med = df_acr_med.drop("teach_sum", "teach_count")

In [0]:
# Create acr field and set it to 'internal medicine'

from pyspark.sql.functions import lit

df_acr_med = df_acr_med.withColumn(
  "acr",
  lit("Internal Medicine")
)

####Non-Medicine ACR Data

In [0]:
# Create a list of column names to be recoded
nonmed_col_labels = [
    "overall_qual", "workload", "teach_faculty", "teach_res"
]

# Create a list of new column names to hold the numeric values
nonmed_recode_labels =  [
   "overall_qual_num", "workload_num", "teach_faculty_num", "teach_res_num"
]

In [0]:
from pyspark.sql.functions import col, when

# Create new numeric columns for each of the specified columns based on satisfaction level
for col_label, recode_label in zip(nonmed_col_labels, nonmed_recode_labels):
  df_acr_nonmed = df_acr_nonmed.withColumn(
    recode_label,
    when(col(col_label) == "Very Satisfied", 5)
    .when(col(col_label) == "Satisfied", 4)
    .when(col(col_label) == "Neutral", 3)
    .when(col(col_label) == "Dissatisfied", 2)
    .when(col(col_label) == "Very Dissatisfied", 1)
    .otherwise(None)  # Use None for unmatched entries to represent missing values
)

In [0]:
# Calculate teach_num by collapsing faculty and resident columns to a single column while accounting for possible null values
from pyspark.sql.functions import col, when, isnull, coalesce

df_acr_nonmed = df_acr_nonmed.withColumn(
    "teach_num",
    when(isnull(col("teach_faculty_num")) & isnull(col("teach_res_num")), None) #when both columns are null, set to null
    .when(isnull(col("teach_faculty_num")), col("teach_res_num")) #when only one column is null, set to the other column
    .when(isnull(col("teach_res_num")), col("teach_faculty_num")) #when only one column is null, set to the other column
    .otherwise((col("teach_faculty_num") + col("teach_res_num")) / 2) #when neither column is null, set to the average of the two columns
)

####Merge Selected Columns from Med and Non-Med Tables

In [0]:
# Select specific columns from df_acr_med
df_acr_med_selected = df_acr_med.select(
    "response_id", "visiting", "curriculum", "acr", "academic_year", 
    "block", "block_num", "released_to_dash", "overall_qual_num", 
    "workload_num", "teach_num"
)

# Select specific columns from df_acr_nonmed
df_acr_nonmed_selected = df_acr_nonmed.select(
    "response_id", "visiting", "curriculum", "acr", "academic_year", 
    "block", "block_num", "released_to_dash", "overall_qual_num", 
    "workload_num", "teach_num"
)

# Combine (union) the two DataFrames
df_all_acrs = df_acr_med_selected.unionByName(df_acr_nonmed_selected)

In [0]:
# Filter to the specified AYs and Blocks by doing an inner join on block and academic_year
filtered_df = df_all_acrs.join(ay_blocks_df, on=["block", "academic_year"], how="inner")

In [0]:
# Filter out visiting student data by removing rows where visiting is 'Yes'
filtered_df = filtered_df.filter(filtered_df.visiting != 'Yes')

In [0]:
# View the filtered DataFrame prepared for analysis
display(filtered_df)

##Calculations and Aggregations

####All ACRS

In [0]:
from pyspark.sql.functions import avg

# Calculate the average values for all ACRs
avg_quality_all = round(filtered_df.agg(avg("overall_qual_num")).collect()[0][0], 2)
avg_workload_all = round(filtered_df.agg(avg("workload_num")).collect()[0][0], 2)
avg_teach_all = round(filtered_df.agg(avg("teach_num")).collect()[0][0], 2)

In [0]:
from pyspark.sql.functions import count

# Calculate the count of responses for all ACRs
count_quality_all = filtered_df.agg(count("overall_qual_num")).collect()[0][0]
count_workload_all = filtered_df.agg(count("workload_num")).collect()[0][0]
count_teach_all = filtered_df.agg(count("teach_num")).collect()[0][0]

####Medicine ACR

In [0]:
# Further filter the DataFrame to include only rows where acr matches the specified value
filtered_to_acr = filtered_df.filter(filtered_df.acr == acr)

# Calculate the average values for the specified ACR
avg_quality_indiv_acr = round(filtered_to_acr.agg(avg("overall_qual_num")).collect()[0][0], 2)
avg_workload_indiv_acr = round(filtered_to_acr.agg(avg("workload_num")).collect()[0][0],2)
avg_teach_indiv_acr = round(filtered_to_acr.agg(avg("teach_num")).collect()[0][0],2)

In [0]:
# Calculate the count of responses for the specified ACR
count_quality_indiv_acr = filtered_to_acr.agg(count("overall_qual_num")).collect()[0][0]
count_workload_indiv_acr = filtered_to_acr.agg(count("workload_num")).collect()[0][0]
count_teach_indiv_acr = filtered_to_acr.agg(count("teach_num")).collect()[0][0]

In [0]:
from pyspark.sql.functions import count, avg

# Define the function to display averages and counts in a clean way
def display_averages_counts(
    avg_quality_all, avg_workload_all, avg_teach_all, count_quality_all, count_workload_all, count_teach_all,
    avg_quality_indiv_acr, avg_workload_indiv_acr, avg_teach_indiv_acr, count_quality_indiv_acr, count_workload_indiv_acr, count_teach_indiv_acr, acr_value
):
    print(f"Averages for All Data:")
    print(f"  Average Overall Quality: {avg_quality_all} (count: {count_quality_all})")
    print(f"  Average Workload: {avg_workload_all} (count: {count_workload_all})")
    print(f"  Average Teaching: {avg_teach_all} (count: {count_teach_all})")
    print()
    print(f"Averages for ACR '{acr_value}':")
    print(f"  Average Overall Quality: {avg_quality_indiv_acr} (count: {count_quality_indiv_acr})")
    print(f"  Average Workload: {avg_workload_indiv_acr} (count: {count_workload_indiv_acr})")
    print(f"  Average Teaching: {avg_teach_indiv_acr} (count: {count_teach_indiv_acr})")

# Display the averages and counts
display_averages_counts(
    avg_quality_all, avg_workload_all, avg_teach_all, count_quality_all, count_workload_all, count_teach_all,
    avg_quality_indiv_acr, avg_workload_indiv_acr, avg_teach_indiv_acr, count_quality_indiv_acr, count_workload_indiv_acr, count_teach_indiv_acr, acr
)
